# 8 盘本地 SSD 扫描
比较 1×、2×、8× 下三档 P 侧 DRAM 的 SLA、分层复用和 SSD 流量。

In [ ]:
from pathlib import Path
import json, re
import pandas as pd
import matplotlib.pyplot as plt

results = Path.home()/'project/results'
batch = results/'glm52-1p1d-local-8ssd/20261106T083401.924673Z'
out = Path.home()/'project/aisimulate/experiments/disk/analysis/p_local_8ssd/assets'
out.mkdir(exist_ok=True)
rows = []
for speed in (1, 2, 8):
    for gb in (128, 256, 384):
        run, = (batch/f'p_local_8ssd_p{gb}gb_{speed}x').glob('*/run.json')
        rows.append((speed, gb, run.parent))

cases, rows = rows, []
for speed, gb, run in cases:
    meta = json.loads((run/'run.json').read_text())
    assert meta['status']=='completed'
    report = run/'prediction/prediction.json'
    with report.open() as f:
        # These summary fields precede the large per-request array.
        header = f.read(25000)
    def metric(name):
        return float(re.search(r'"'+name+r'"\s*:\s*([0-9.eE+-]+)',header)[1])
    g3 = json.JSONDecoder().raw_decode(header[re.search(r'"g3_offload"\s*:\s*',header).end():])[0]
    completed = int(metric('completed_requests'))
    assert completed == meta['selected_requests'] == 665415
    good = int(metric('goodput_completed_requests'))
    totals = [0, 0, 0]
    with report.with_name('requests.jsonl').open() as stream:
        for line in stream:
            req = json.loads(line)
            for i, key in enumerate(('input_length', 'first_admission_g2_direct_reused_input_tokens',
                                     'first_admission_g3_reused_input_tokens')):
                totals[i] += req[key]
    total_pct = 100*metric('first_admission_prefix_cache_reused_ratio')
    g2_pct, g3_pct = (100*t/totals[0] for t in totals[1:])
    rows.append(dict(speedup=speed,dram_gb=gb,g1_pct=total_pct-g2_pct-g3_pct,
                     g2_pct=g2_pct,g3_pct=g3_pct,total_reuse_pct=total_pct,completed=completed,sla_pass=good,sla_percent=100*good/completed,
                     mean_ttft_s=metric('mean_ttft_ms')/1000,p95_ttft_s=metric('p95_ttft_ms')/1000,
                     p99_ttft_s=metric('p99_ttft_ms')/1000,max_gap_s=metric('max_itl_ms')/1000,
                     ssd_read_tb=g3['read']['completed_bytes']/1e12,ssd_write_tb=g3['write']['completed_bytes']/1e12,
                     g3_bypassed_restores=g3.get('bypassed_restores',0),
                     trace_sha256=meta['selected_trace_sha256'],report=str(report)))
assert len({row['trace_sha256'] for row in rows})==1
frame = pd.DataFrame(rows)
frame.to_csv(out/'metrics.csv',index=False)
(out/'summary.json').write_text(json.dumps(rows,indent=2))
print(frame.drop(columns=['report','trace_sha256']).to_string(index=False))
fig, axes = plt.subplots(1,3,figsize=(12,3.8))
for ax,key,label in zip(axes,['sla_percent','ssd_read_tb','max_gap_s'],
                       ['SLA pass (%)','SSD read (TB)','Max token gap (s)']):
    for speed, group in frame.groupby('speedup'):
        ax.plot(group.dram_gb,group[key],marker='o',label=f'{speed}x')
    ax.legend()
    ax.set(xlabel='P DRAM (GB)',ylabel=label)
    ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(out/'comparison.png',dpi=160)
plt.show()
